# 02 · Análisis de identidad

**Proyecto:** Spotify Music Intelligence
**Módulo 2:** Limpieza, catálogo e identidad
**Objetivo:** revisar la consolidación por `track_id`, la generación exacta de
`recording_group_id` y el reporte de candidatos a casi duplicados, sin modificar
ningún dato.

## Contexto

El dataset original tiene 114.000 filas y 89.741 `track_id` únicos (24.259 filas
adicionales). Una fila con identidad inválida se envía a cuarentena, por lo que
quedan **89.740** `track_id` válidos en `tracks.parquet`.

La huella exacta combina `track_name_normalized`, `artists_normalized` y 14
valores de audio (sin redondeo) y se serializa como SHA-256:
`recording_group_id` de 64 caracteres hexadecimales.

Reglas clave:

- No se reescribe ningún `track_id`.
- La agrupación exacta es conservadora: no elimina `live`, `remix`, etc.,
  no ordena artistas y no elimina puntuación.
- Los casi duplicados **no se fusionan automáticamente**; solo se reportan.

## Configuración y carga

Se cargan los datos procesados. El cálculo descriptivo es fino y no modifica
archivos.

In [1]:
import os
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

tracks = pd.read_parquet("data/processed/tracks.parquet")
recordings = pd.read_parquet("data/processed/recordings.parquet")
recording_tracks = pd.read_parquet("data/processed/recording_tracks.parquet")
candidates = pd.read_csv("reports/identity/near_duplicate_candidates.csv")

print("tracks:", tracks.shape)
print("recordings:", recordings.shape)
print("recording_tracks:", recording_tracks.shape)
print("candidates:", candidates.shape)

tracks: (89740, 29)
recordings: (83881, 24)
recording_tracks: (89740, 2)
candidates: (69556, 9)


# 1. Consolidación y grupos exactos

## 1.1 ¿Cuántas grabaciones se consolidan por `track_id`?

**Método:** comparación de los conteos del manifiesto del pipeline y conteos
derivados de los datos procesados.

In [2]:
print("Track IDs válidos:", tracks["track_id"].nunique())
print("Recording groups:", tracks["recording_group_id"].nunique())
print("Filas en recording_tracks:", len(recording_tracks))
print("Grabaciones en recordings:", len(recordings))
print(
    "Tracks que comparten grupo con otro (>1):",
    int(tracks["recording_group_id"].duplicated(keep=False).sum()),
)

Track IDs válidos: 89740
Recording groups: 83881
Filas en recording_tracks: 89740
Grabaciones en recordings: 83881
Tracks que comparten grupo con otro (>1): 8589


**Resultado:** los conteos coinciden con el manifiesto del pipeline
(89.740 `track_id` y 83.881 `recording_group_id`).

**Interpretación:** la consolidación es moderada: la mayoría de las pistas forma
un grupo propio, pero 2.730 grupos agrupan más de un `track_id`. Ninguna ID queda
huérfana: toda fila de `tracks` tiene su grupo y toda fila de `recording_tracks`
pertenece a un grupo presente en `recordings`.

**Limitación:** la huella exacta es conservadora por diseño; los casos límite se
estudian por separado como candidatos a casi duplicados.

## 1.2 ¿Cómo se distribuye el tamaño de los grupos?

**Método:** distribución de `track_id` por `recording_group_id` y grupos más
grandes.

In [3]:
group_sizes = tracks.groupby("recording_group_id").size()
print("Grupos de tamaño 1:", int((group_sizes == 1).sum()))
print("Grupos con más de un track:", int((group_sizes > 1).sum()))
print("Tamaño máximo:", int(group_sizes.max()))
print("Grupos con ≥ 10 tracks:", int((group_sizes >= 10).sum()))
print("\nDistribución (tamaño -> nº de grupos):")
group_sizes.value_counts().sort_index().head(12)

Grupos de tamaño 1: 81151
Grupos con más de un track: 2730
Tamaño máximo: 42
Grupos con ≥ 10 tracks: 97

Distribución (tamaño -> nº de grupos):


1     81151
2      1788
3       430
4       181
5        91
6        60
7        36
8        30
9        17
10       14
11        8
12       13
Name: count, dtype: int64

In [4]:
largest = (
    tracks.assign(tamaño_grupo=tracks["recording_group_id"].map(group_sizes))
    .sort_values(["tamaño_grupo", "popularity_median"], ascending=[False, False])
    .drop_duplicates("recording_group_id")[
        ["recording_group_id", "track_name", "artists", "tamaño_grupo", "popularity_median"]
    ]
    .head(10)
)
largest

,recording_group_id,track_name,artists,tamaño_grupo,popularity_median
25360,4469351bf23323c0602e54fb28af1bef609d43a21f766d...,Rockin' Around The Christmas Tree,Brenda Lee,42,1.0
5059,7f5b0508b7f59c7011c8311cbfc88dfa0b0f20294fe8b2...,Little Saint Nick - 1991 Remix,The Beach Boys,41,0.5
3238,40d07dcdbd2c7256eff762ac74276e04092f244959dbf0...,Run Rudolph Run,Chuck Berry,38,1.0
7499,236c63d8167317b3f9efb011597ddc6f51cf5144b1ea56...,Frosty The Snowman,Ella Fitzgerald,34,4.0
4596,73c844c9b5be8545b5fb389d2d5b2e98636feea2437468...,Mistletoe,Justin Bieber,31,4.0
3937,6025f01131d31c609876ac74800c237d839ed005f8ecd3...,Sleigh Ride,Ella Fitzgerald,30,3.0
2311,27cc5c3e5cd02667f66d602d43122fb7f12ee5522ff46d...,I Saw Mommy Kissing Santa Claus,The Jackson 5,27,2.0
63859,ca607ca6c47f559acc0185cd9a92d5e9a1ca8166710c67...,Last Last,Burna Boy,26,83.0
30029,674a075d6fd4c34257c45a0b8c3bc826de4ed1e5362f7f...,Let It Snow! Let It Snow! Let It Snow!,Dean Martin,26,3.0
2986,03ff0d9e7ad01474150266395112e76c82a84b70e330a2...,Santa Claus Is Coming To Town,The Jackson 5,26,0.0


**Resultado:** 81.151 grupos (96,7 %) tienen un único track y 2.730
(3,3 %) agrupan varias pistas, con un máximo de 42. Los grupos más grandes
corresponden a pistas repetidas en el catálogo bajo metadatos idénticos o
muy próximos.

**Limitación:** un tamaño grande de grupo no implica que sean versiones oficiales
equivalentes; solo que la huella exacta coincide.

## 1.3 ¿Qué track representa a cada grupo?

**Método:** comprobación de la regla de: mayor `popularity_median`
y, en empate, menor `track_id` lexicográfico.

In [5]:
recordings[["recording_group_id", "representative_track_id", "track_name", "artists"]].head(8)

,recording_group_id,representative_track_id,track_name,artists
0,000110937061adb7eb0566ed2b6f530764bf45453de4ba...,0JPHV6kJG2QB7XUXH9eqCk,Bye Bye Bye (feat. Chronixx) - S.P.Y Remix,SHY FX;Chronixx;S.P.Y
1,0001dbfc8189f850c8f76a5b815a3b53511da67a2022d4...,49KVtxjMNy8li0sQt4exTD,Arcadia,Apparat
2,0004691fa45adbd304ec50172b0f657c3a5f5802ed18a7...,4a546ManD3y500ovANiMEY,Relay Goodbye,Casino Versus Japan
3,000475e18edcbe5af041b20a7c581c8bda6ba9dc8ab1a4...,4IpJwePXEWjNf9IrWAvcGg,Ecstasy,Birat Bitz
4,000515f36c963ec1cdc61bee0f31d8bf5aecb90d6fe739...,5FkEhuXoU6xk7x6xtmkG1z,Leef Voor `t Weekend,PartyfrieX;DJ Carlo
5,0005efa781d309ee49a5663a76f8c4b5d892e1705f2c36...,2W9MxlcyMWmeHgZNnCSosb,污糟兒,Kaho Hung
6,0006a4445fe23b62c5c8c338ce8c7f23eecd1f485ab69e...,25k1Qv4vHmop3ZG69tiK0U,Define God,Psyko Punkz
7,0008269a7bfb43832b6879459d4689c08e0d6928c92ed4...,3XM2ormuVHvn3tQvTQoy3J,Circles - 7'' Edit,Adam F


**Interpretación:** `recordings.parquet` contiene exactamente una fila por
grupo con el track representativo elegido por la regla de popularidad. Esta fila
es la que se muestra en recomendadores y clasificadores.

**Limitación:** la selección no afirma que sea la versión oficial o más reciente;
es solo una regla determinista para presentar el grupo.

# 2. Candidatos a casi duplicados

## 2.1 ¿Cuántos candidatos y de qué tipo?

**Método:** conteo de pares por tipo de evidencia (`textual` o `acoustic`) en
`reports/identity/near_duplicate_candidates.csv`.

In [6]:
print("Total de pares candidatos:", len(candidates))
print(candidates["evidence"].value_counts().to_string())

Total de pares candidatos: 69556
evidence
acoustic    58813
textual     10743


**Resultado:** 69.556 pares candidatos, de los cuales 58.813 proceden de
la señal acústica y 10.743 de la señal textual.

**Interpretación:** la señal acústica (umbral de similitud 0,98 en el espacio
normalizado) es la fuente principal de candidatos. La textual requiere compartir
la clave de título flexible normalizada y estar dentro de la tolerancia de
duración, por lo que es más selectiva en pares pero aporta parejas que la señal
acústica puede no haber captado.

**Limitación:** estos pares son **candidatos**, no duplicados confirmados. La
fusión solo puede realizarse tras revisión manual y una regla versionada.

## 2.2 ¿Qué tan similares son los candidatos textuales?

**Método:** distribución de `title_similarity` (RapidFuzz ratio) sobre los pares
con evidencia textual.

In [7]:
textual = candidates[candidates["evidence"] == "textual"]
print(textual["title_similarity"].describe().to_string())
print("\nBins de similitud de título:")
pd.cut(textual["title_similarity"], bins=[0, 60, 80, 90, 99, 100]).value_counts().sort_index()

count    10743.000000
mean        95.463751
std         16.086402
min          0.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000

Bins de similitud de título:


title_similarity
(0, 60]       475
(60, 80]      164
(80, 90]      328
(90, 99]      648
(99, 100]    9114
Name: count, dtype: int64

In [8]:
textual.nlargest(10, "title_similarity")[
    [
        "recording_group_id_a",
        "recording_group_id_b",
        "track_name",
        "artists_a",
        "artists_b",
        "duration_diff_ms",
        "title_similarity",
    ]
]

,recording_group_id_a,recording_group_id_b,track_name,artists_a,artists_b,duration_diff_ms,title_similarity
60,6de5b76bfd5d1b87aa6264bd3af0b1127fde71b94bd4b4...,d0ac8cc6cc63817b363dd18ce0012d4c37c18443b38819...,(Don't Fear) The Reaper,Blue Öyster Cult,The Countdown Kids,3852,100.0
61,991dfb3d61e73fa7bf235e693cc8d63849867d3c05e677...,de848353f55e789184ab9711217072c5ed78339bd62bb9...,(Everything I Do) I Do It For You,Bryan Adams,Bryan Adams,493,100.0
66,7dd4f4adf8c92809d07aeec6beafba825add1c6bb59ad7...,fc51cf272f021e5ffcf4a32a152bdde11679e2a82458f5...,(I Can't Get No) Satisfaction - Mono Version,The Rolling Stones,The Rolling Stones,67,100.0
73,286e5e23112a020b41eda948031964c25471ffd1afb522...,58378e76491128ef6878adb9cbdcf5533d8a06ebe9d024...,(I Heard That) Lonesome Whistle - Single Version,Hank Williams;Drifting Cowboys,Hank Williams;Drifting Cowboys,1533,100.0
80,1d2484de49b4de41d9de018f7ace5adff70e93d27b452d...,7f117db8ddab52bd69d5cbe94d792cdeed6f42dab54786...,(I Just) Died In Your Arms,Cutting Crew,Cutting Crew,1774,100.0
81,1d2484de49b4de41d9de018f7ace5adff70e93d27b452d...,e533a9081a91e4ef8c01f0fd9f8dbd3e477ddc942b33f2...,(I Just) Died In Your Arms,Cutting Crew,Cutting Crew,11360,100.0
82,7f117db8ddab52bd69d5cbe94d792cdeed6f42dab54786...,e533a9081a91e4ef8c01f0fd9f8dbd3e477ddc942b33f2...,(I Just) Died In Your Arms,Cutting Crew,Cutting Crew,9586,100.0
100,1f604c380f0c3730fe5ade15d5d0abf1d9b92395161d27...,d550e86f1f6b6fbcee0645d93cfc2c5134b45723679423...,(We're Not) The Jet Set,George Jones;Tammy Wynette,George Jones;Tammy Wynette,707,100.0
101,1f604c380f0c3730fe5ade15d5d0abf1d9b92395161d27...,f3b023647a36520d367b8a60d6baa6beb7e9314ffa74dd...,(We're Not) The Jet Set,George Jones;Tammy Wynette,George Jones;Tammy Wynette,374,100.0
102,d550e86f1f6b6fbcee0645d93cfc2c5134b45723679423...,f3b023647a36520d367b8a60d6baa6beb7e9314ffa74dd...,(We're Not) The Jet Set,George Jones;Tammy Wynette,George Jones;Tammy Wynette,333,100.0


**Resultado:** el 85 % de los pares textuales supera 99 de similitud y la
mediana es 100. Los pares con similitud baja (0) corresponden a títulos distintos
que comparten clave normalizada solo por signos de puntuación o espacios.

**Limitación:** el ratio de RapidFuzz es una medida de edición; dos canciones
distintas pueden compartir clave flexible (p. ej. títulos con signos) y generar
falsos positivos que la revisión manual debe descartar.

## 2.3 ¿Qué tan similares son los candidatos acústicos?

**Método:** distribución de `acoustic_similarity` sobre los pares con evidencia
acústica (similaridad coseno en el espacio normalizado de 9 características).

In [9]:
acoustic = candidates[candidates["evidence"] == "acoustic"]
print(acoustic["acoustic_similarity"].describe().to_string())
print("\nBins de similitud acústica:")
pd.cut(
    acoustic["acoustic_similarity"], bins=[0.98, 0.99, 0.995, 0.999, 1.001]
).value_counts().sort_index()

count    58813.000000
mean         0.987402
std          0.004960
min          0.980000
25%          0.983294
50%          0.986702
75%          0.990816
max          1.000000

Bins de similitud acústica:


acoustic_similarity
(0.98, 0.99]      41522
(0.99, 0.995]     11905
(0.995, 0.999]     4563
(0.999, 1.001]      819
Name: count, dtype: int64

In [10]:
acoustic.nlargest(10, "acoustic_similarity")[
    [
        "recording_group_id_a",
        "recording_group_id_b",
        "track_name",
        "artists_a",
        "artists_b",
        "duration_diff_ms",
        "acoustic_similarity",
    ]
]

,recording_group_id_a,recording_group_id_b,track_name,artists_a,artists_b,duration_diff_ms,acoustic_similarity
153,5610510557beb897d76f6db76f0e6c30ad28e102ade684...,877e0715d2875f306021d6ae76303b3652b9f650b78a88...,"1, 2, 3",Sofía Reyes;Jason Derulo;De La Ghetto,Sofía Reyes;Jason Derulo;De La Ghetto,0,1.0
182,75d23843b151d72d4310e06468013e17b55e9e8ecd5b21...,7a220494bcfb51a1360ef070122cd260d0351c6e34b335...,"10,000 Hours (with Justin Bieber)",Dan + Shay;Justin Bieber,Dan + Shay;Justin Bieber,0,1.0
417,582d39e42181664dde7ab8f1450fc7b1d373d04d91c9bf...,9f21f34f3370225ae8e2020a5672d295831847e8bab7e7...,21 Reasons (feat. Ella Henderson),Nathan Dawe;Ella Henderson,Nathan Dawe;Ella Henderson,0,1.0
627,2b33b76de07f31e3da73043daf51c43e64bcc05ae73ac6...,e2020929744aa0fc8ad7e04d624a7256e3bc98101ace0e...,679,Fetty Wap;Remy Boyz,Fetty Wap;Remy Boyz,0,1.0
1434,096f130aa13984cf644fe052f1e2c8ae42bccc52ababbb...,bbb52786f1ce8551caa3e8e3922846b6af2fef70e9afe5...,"Aabaad Barbaad (From ""Ludo"")",Pritam;Arijit Singh,Pritam;Arijit Singh,0,1.0
1471,3133b3bb4880f6b3008fa0d7ccb2a28e96fac919dd1151...,5e010df5190b602a6365ec63ed4be7600e73026cf72974...,Aaro Aaro Chare,Najim Arshad,Najim Arshad,0,1.0
1478,a34b5588d65a40d23339b4ed1473e025ff774455f087ea...,ab9151deda5ca469b065ef7e8cfecb1a55f9a925712065...,"Aashiqui Aa Gayi(From ""Radhe Shyam"")",Mithoon;Arijit Singh,Mithoon;Arijit Singh,0,1.0
1480,4fdbc65c5eeff1d31325958ddf2bbe31597f70994ec4ea...,73a950a04eccd544042773389ba9fb2fbfbe4cb148331a...,"Aathadi Manasudhan (From ""Kazhugoo"")",Yuvan Shankar Raja;Priya Hemesh,Yuvan Shankar Raja;Priya Hemesh,0,1.0
1482,ba1ea12e4f9103b4dcd9c0bf46fe6652bfa7207c8e3221...,f4958bcd69fdef375c05b9d53135df357a6bef66ca6b1e...,Aattinkarayorathe,Manjari,Manjari,0,1.0
1684,0c9ffc5f950aaa05d0843faf338414322a5aaaa8354d23...,12abe04ffa7e40f43830870a0752a5f56e05ad7ce31e83...,"Adada Mazhaida (From ""Paiya"")",Yuvan Shankar Raja;Rahul Nambiar;Saindhavi,Yuvan Shankar Raja;Rahul Nambiar;Saindhavi,0,1.0


**Resultado:** la similitud acústica de los candidatos se concentra entre
0,98 y 0,99 (70 % de los pares); solo 819 pares superan 0,999. La similaridad
media es 0,987.

**Limitación:** la similaridad coseno no es una probabilidad de duplicidad ni de
gusto. Un par puede ser muy similar acústicamente y ser
una grabación distinta (p. ej. versión o remezcla).

## Conclusión del análisis de identidad

**Resumen:** la consolidación es correcta y verificable: 89.740 `track_id` →
83.881 `recording_group_id`, sin IDs huérfanas. El 96,7 % de los grupos es
individual y el 3,3 % restante agrupa pistas de huella idéntica. El reporte de
candidatos (69.556 pares) queda como **insumo de revisión manual**; no se realiza
ninguna fusión automática.

**Siguiente paso:** usar `recordings.parquet` como unidad de modelado en los
módulos 4 y 6, y revisar los candidatos antes de definir cualquier regla de
fusión versionada.

Limitaciones: la huella exacta es conservadora; el reporte de candidatos contiene
falsos positivos esperados y no constituye duplicidad confirmada.